# Training Pipeline — ResNet-50

The same pipeline as [`_templates/Evaluation_Template.ipynb`](../../../_templates/Evaluation_Template.ipynb)
(itself a cell-by-cell expansion of MedViTV2's
[`Tutorials/Evaluation.ipynb`](https://github.com/Omid-Nejati/MedViTV2/blob/main/Tutorials/Evaluation.ipynb)),
but training a **ResNet-50** instead of a MedViT. Loss selection, optimizer, schedule, training
loop and metrics are unchanged, so the two runs stay directly comparable.

**No MedViTV2 clone.** The template cloned that repo for two things — the `MedViT_*` builders and
`datasets.py`. Neither is needed here: `'resnet50'` is built by [timm](https://timm.fast.ai/), and
the MedMNIST half of `datasets.py` is ported into
[`_handlers/datasets.py`](../../../_handlers/datasets.py) with the same transforms and the same
`medmnist` calls. The `natten` install goes too — it was a MedViT-only dependency. What *is* cloned
is **this** repository, so the `_handlers` package is importable when running on Colab.

The other **reusable pieces** (model selection, the training routine, the metric helpers and the
all-metrics evaluation) live in [`_handlers/evaluation.py`](../../../_handlers/evaluation.py); the
notebook keeps only the configuration and the linear flow.

Use a GPU (a Colab **T4** is enough) — `build_model` calls `.cuda()` on the network.

## Install Requirements

Only what the pipeline actually imports: `medmnist` for the data, `timm` for the model,
`scikit-learn` for the metrics, plus `tqdm`/`requests`.

In [ ]:
!nvidia-smi

In [ ]:
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!pip install timm medmnist==3.0.2 scikit-learn tqdm requests

## Get the survey code

The pipeline lives in this repository's `survey/src/_handlers` package, so the notebook needs a
checkout of it. On Colab the working directory is `/content` and nothing is there yet — the cell
below clones the repo and points `SRC` at `survey/src`. Run locally, the notebook already sits
inside the repo, so the climb finds `_handlers` immediately and nothing is cloned.

If the repository is private the anonymous clone fails with an authentication error; use a token
URL instead — `REPO_URL = 'https://<GITHUB_TOKEN>@github.com/alexandrachirita98/quantum-quantization.git'`.

In [ ]:
import pathlib
import subprocess

REPO_URL = 'https://github.com/alexandrachirita98/quantum-quantization.git'


def find_src(start):
    """Climb from `start` looking for the survey `src/` root — the directory holding `_handlers`."""
    for p in [pathlib.Path(start), *pathlib.Path(start).parents]:
        if (p / '_handlers').is_dir():
            return p
    return None


SRC = find_src(pathlib.Path.cwd())
if SRC is None:                                          # Colab: no checkout yet, clone one
    clone_dir = pathlib.Path.cwd() / 'quantum-quantization'
    if not clone_dir.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(clone_dir)], check=True)
    SRC = clone_dir / 'survey' / 'src'

assert (SRC / '_handlers').is_dir(), f'no _handlers package under {SRC}'
print('survey src:', SRC)

## Imports

`INFO` comes straight from the `medmnist` package (it carries each dataset's `task` and label
map); `build_dataset` and the training/metric helpers come from the survey's `_handlers`.

`SRC` from the previous cell goes on `sys.path`, and `os.chdir` moves into it so the `./data`
folder the pipeline reads and writes is the shared [`src/data`](../../../data) — the same path the
medmnist `Evaluator` uses inside `evaluation.py`.

In [ ]:
import os
import sys

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from medmnist import INFO

# import the pipeline from the survey `src/` root, and work from there so './data' resolves inside it
sys.path.insert(0, str(SRC))
os.chdir(SRC)
print('working directory:', os.getcwd())

from _handlers.datasets import build_dataset
from _handlers.evaluation import (
    build_model,
    train_mnist,
    evaluate_all_metrics,
)

## Configuration

`main.py` reads its settings from `argparse` on the command line. Here we replace that block with a
plain config object, so the exact same `args` flows through the pipeline.

**Model** — `resnet50`, resolved by timm. Any other timm name works the same way (`resnet18`,
`resnet101`, `convnext_tiny`, …); with `pretrained=True` timm downloads the ImageNet weights and
swaps in a fresh `nb_classes` head. Left at `False` to match the template's train-from-scratch
setting.

**Dataset** — any MedMNIST flag: `tissuemnist, pathmnist, chestmnist, dermamnist, octmnist,
pneumoniamnist, retinamnist, breastmnist, bloodmnist, organamnist, organcmnist, organsmnist`. The
first download can take a while. The paper's image-folder datasets (`Kvasir`, `CPN`, `Fetal`,
`PAD`, `ISIC2018`) are *not* available without the upstream repo — their downloaders were not
ported.

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    model_name='resnet50',                           # any timm model name
    dataset='breastmnist',                           # a medmnist flag
    batch_size=24,
    lr=1e-4,
    epochs=100,
    pretrained=False,                                # load timm's ImageNet weights
    checkpoint_path=None,                            # unused on the timm path
)

## Device

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using {} device.".format(device))

## Dataset

The `task` field from `INFO` (multi-label vs. multi-class) selects the loss. `build_dataset` then
downloads the `.npz` into `./data`, serves it as 3-channel 224×224, applies the transforms and
returns the datasets plus the number of classes.

In [ ]:
# the medmnist `task` selects the loss
info = INFO[args.dataset]
task = info['task']
if task == "multi-label, binary-class":
    loss_function = nn.BCEWithLogitsLoss()
else:
    loss_function = nn.CrossEntropyLoss()

train_dataset, test_dataset, nb_classes = build_dataset(args=args)

print(train_dataset)
print("===================")
print(test_dataset)

## Model

`build_model` builds a `MedViT_*` only when the name is a key of `model_classes`; every other name
falls through to `timm.create_model`. Here `model_classes` is **empty**, so `'resnet50'` goes
straight to timm and comes back with an `nb_classes` head, on CUDA. Swapping `args.model_name` for
another timm name is the only change needed to train a different backbone.

In [ ]:
model_classes = {}   # no MedViT builders — every name falls through to timm

net = build_model(args.model_name, nb_classes, model_classes,
                  pretrained=args.pretrained, checkpoint_path=args.checkpoint_path)

## Optimizer, Scheduler & Data Loaders

AdamW with weight decay and a cosine-annealing schedule stepped **every iteration** — so `T_max` is
the total number of optimizer steps (`epochs * train_num // batch_size`).

In [ ]:
train_num = len(train_dataset)
eta = args.epochs * train_num // args.batch_size   # total scheduler steps

optimizer = optim.AdamW(net.parameters(), lr=args.lr, betas=[0.9, 0.999], weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=eta, eta_min=5e-6)

train_loader = data.DataLoader(dataset=train_dataset, batch_size=args.batch_size, shuffle=True)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=2*args.batch_size, shuffle=False)

## Train

`train_mnist` (from the handler) runs the loop, scoring every epoch with the medmnist `Evaluator`
(AUC/ACC) and writing the best model to `save_path`.

In [ ]:
save_path = f'./{args.model_name}_{args.dataset}.pth'

train_mnist(args.epochs, net, train_loader, test_loader,
            optimizer, scheduler, loss_function, device, save_path, args.dataset, task)

## Evaluate all metrics

`evaluate_all_metrics` (from the handler) runs the model once over one split and prints **every**
metric the training routines can produce: the medmnist Evaluator AUC/ACC plus accuracy, weighted
precision / recall (sensitivity) / F1, per-class + average specificity, one-vs-rest AUC, the
confusion matrix and a per-class report. Set `split` to `'train'` or `'test'`; uncomment the
`load_state_dict` line to score the best checkpoint instead of the in-memory model.

In [ ]:
split = 'test'   # 'train' or 'test'

# net.load_state_dict(torch.load(save_path)['model'])   # uncomment to evaluate the BEST checkpoint

eval_dataset = train_dataset if split == 'train' else test_dataset
metrics = evaluate_all_metrics(net, eval_dataset, args.dataset, nb_classes, device,
                               split=split, batch_size=2 * args.batch_size)